# Data Preparation

Validate normalized records, split manifests, and audio-level leakage.

## Rebuild manifests from raw metadata (authoritative, reproducible)

`scripts/build_manifests.py` is the single source of truth for
`dataset/dcase_part1/splits/{train,validation,test}.jsonl`. It discovers the
raw per-record metadata, normalizes it, assigns a deterministic 90/10
audio-level train/validation split, and removes any audio that also appears
in the official dev/test partition from train and validation (test is never
modified). Run with `--dry-run` first to preview counts without writing.

In [ ]:
import subprocess, sys

# Dry run only inside the notebook -- rerun without --dry-run from a shell
# when you actually want to regenerate the checked-in manifests.
result = subprocess.run(
    [sys.executable, "scripts/build_manifests.py", "--dry-run"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr)

In [ ]:
from src.data.dataset_loader import load_all_splits
from src.data.splitting import check_audio_leakage

splits = load_all_splits()
records = {k: [r.to_dict() for r in v] for k, v in splits.items()}
check_audio_leakage(records)

In [ ]:
summary = {k: {'records': len(v), 'audio_files': len({r.audio_path for r in v})} for k, v in splits.items()}
summary

In [ ]:
import json
from pathlib import Path

Path('dataset/dcase_part1/splits/summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary